In [ ]:
import pandas as pd

pd.set_option(
    "display.float_format",
    lambda x: f"{x:.4f}".rstrip("0").rstrip(".")
)

# Pipeline 產出物檢視

手動檢視 pipeline 執行後的產出物（parquet、pickle、json），用於除錯和驗證資料品質。

In [ ]:
"""設定環境：載入配置與建立 DataCatalog"""
import os
from pathlib import Path
os.chdir(os.path.join(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()))

from recsys_tfb.core.config import ConfigLoader
from recsys_tfb.core.catalog import DataCatalog
from recsys_tfb.core.versioning import resolve_dataset_version, resolve_model_version

config = ConfigLoader("conf", env="local")
data_dir = Path("data")

# 解析版本
dataset_version = resolve_dataset_version(data_dir / "dataset", None)
model_version = resolve_model_version(data_dir / "models", None)

try:
    params_inf = config.get_parameters_by_name("parameters_inference")
    inf_config = params_inf.get("inference", params_inf)
    snap_dates = inf_config.get("snap_dates", [])
    snap_date = snap_dates[0].replace("-", "") if snap_dates else "unknown"
except KeyError:
    snap_date = "unknown"

runtime_params = {
    "dataset_version": dataset_version,
    "model_version": model_version,
    "snap_date": snap_date,
}

# 建立 catalog，model/preprocessor 透過 best symlink 讀取（與 __main__.py 一致）
catalog_config = config.get_catalog_config(runtime_params=runtime_params)
for entry_name in ("model", "preprocessor"):
    if entry_name in catalog_config:
        catalog_config[entry_name]["filepath"] = catalog_config[entry_name]["filepath"].replace(model_version, "best")
catalog = DataCatalog(catalog_config)

print(f"Dataset version: {dataset_version}")
print(f"Model version: {model_version}")
print(f"Snap date: {snap_date}")
print("\n已註冊的資料集：")
for name in catalog.list():
    status = "存在" if catalog.exists(name) else "不存在"
    print(f"  - {name} ({status})")

## Source Data

In [ ]:
"""檢視 feature_table（Parquet）"""
if catalog.exists("feature_table"):
    df = catalog.load("feature_table")
    print(f"Shape: {df.shape}")
    print(f"\nDtypes:\n{df.dtypes}")
    print(f"\nCount cust_id:\n{df.cust_id.nunique()}")
    print(f"\nCount snap_dates:\n{df.snap_date.unique()}")
    print(f"\nNull 統計:\n{df.isnull().sum()}")
    display(df.head(10))
    display(df.describe())
else:
    print("feature_table 不存在，請先執行 dataset building pipeline。")

In [ ]:
"""檢視 label_table（Parquet）"""
if catalog.exists("label_table"):
    df = catalog.load("label_table")
    print(f"Shape: {df.shape}")
    print(f"\nDtypes:\n{df.dtypes}")
    print(f"\nCount cust_id:\n{df.cust_id.nunique()}")
    print(f"\nCount snap_date:\n{df.snap_date.unique()}")
    print(f"\nCount prod_name:\n{df.prod_name.nunique()}")
    print(f"\nNull 統計:\n{df.isnull().sum()}")
    display(df.head(10))
    display(df.describe())

    # Label 分佈
    if "label" in df.columns:
        print(f"\nLabel 分佈:\n{df['label'].value_counts()}")
    if "prod_name" in df.columns:
        print(f"\nProd_name 分佈:\n{df['prod_name'].value_counts()}")
        # 各產品正樣本比例
        pos_rate = df.groupby("prod_name")["label"].mean().sort_values(ascending=False)
        print(f"\n各產品正樣本比例:\n{pos_rate}")
    if "snap_date" in df.columns:
        print(f"\nSnap_date 分佈:\n{df['snap_date'].value_counts().sort_index()}")
else:
    print("label_table 不存在，請先執行 dataset building pipeline。")

## Dataset Pipeline

In [ ]:
"""載入 train_set、train_dev_set、val_set 並顯示 shape"""
import pandas as pd

splits = {}
for name in ["train_set", "train_dev_set", "val_set"]:
    if catalog.exists(name):
        splits[name] = catalog.load(name)
        print(f"{name}: {splits[name].shape}")
    else:
        print(f"{name} 不存在，請先執行 dataset pipeline。")

In [ ]:
"""檢視 split keys：sample_keys、train_keys、train_dev_keys、val_keys"""
for name in ["sample_keys", "train_keys", "train_dev_keys", "val_keys"]:
    if catalog.exists(name):
        df = catalog.load(name)
        print(f"{name}: {df.shape}")
        print(f"  Columns: {list(df.columns)}")
        print(f"  count cust_id: {df.cust_id.nunique()}")
        print(f"  snap_date: {df.snap_date.unique()}")
        display(df.head(5))
    else:
        print(f"{name} 不存在，請先執行 dataset pipeline。")
    print()

In [ ]:
"""基本統計比較：數值特徵 describe() — splits vs source feature_table"""
num_cols = ["total_aum", "fund_aum", "in_amt_sum_l1m", "out_amt_sum_l1m",
            "in_amt_ratio_l1m", "out_amt_ratio_l1m"]

source_ft = catalog.load("feature_table")

desc_parts = pd.concat({"source_feature_table": source_ft[num_cols].describe()}, axis=1)
display(desc_parts)

for name, df in splits.items():
    desc_parts = {}
    cols = [c for c in num_cols if c in df.columns]
    if cols:
        desc_parts[name] = df[cols].describe()

    desc_parts = pd.concat(desc_parts, axis=1)
    display(desc_parts)

In [ ]:
"""Label 分佈比較：各 split 正樣本率 vs source label_table"""
source_lt = catalog.load("label_table")

# Overall 正樣本率
overall_rates = {"source_label_table": source_lt["label"].mean()}
for name, df in splits.items():
    if "label" in df.columns:
        overall_rates[name] = df["label"].mean()
print("Overall 正樣本率：")
for k, v in overall_rates.items():
    print(f"  {k}: {v:.4f}")

# Per-product 正樣本率
print("\nPer-product 正樣本率：")
per_prod_parts = {"source_label_table": source_lt.groupby("prod_name")["label"].mean()}
for name, df in splits.items():
    if "label" in df.columns and "prod_name" in df.columns:
        per_prod_parts[name] = df.groupby("prod_name")["label"].mean()

per_prod = pd.DataFrame(per_prod_parts)
display(per_prod)

In [ ]:
"""檢視 preprocessed arrays：X_train/y_train、X_train_dev/y_train_dev、X_val/y_val"""
import numpy as np

for prefix in ["train", "train_dev", "val"]:
    x_name = f"X_{prefix}"
    y_name = f"y_{prefix}"
    if catalog.exists(x_name):
        X = catalog.load(x_name)
        if isinstance(X, np.ndarray):
            print(f"{x_name}: shape={X.shape}, dtype={X.dtype}")
        else:
            print(f"{x_name}: shape={X.shape}, type={type(X).__name__}")
            print(f"  Dtypes:\n{X.dtypes}")
    else:
        print(f"{x_name} 不存在，請先執行 dataset pipeline。")
    if catalog.exists(y_name):
        y = catalog.load(y_name)
        if isinstance(y, np.ndarray):
            print(f"{y_name}: shape={y.shape}, dtype={y.dtype}")
            if y.size > 0:
                print(f"  正樣本率: {y.mean():.4f}")
        else:
            print(f"{y_name}: shape={y.shape}, type={type(y).__name__}")
            if hasattr(y, 'mean'):
                mean_val = y.mean()
                if hasattr(mean_val, 'item'):
                    mean_val = mean_val.item()
                print(f"  正樣本率: {mean_val:.4f}")
    else:
        print(f"{y_name} 不存在，請先執行 dataset pipeline。")
    print()

## Training Pipeline

In [ ]:
"""檢視 category_mappings（JSON）"""
if catalog.exists("category_mappings"):
    mappings = catalog.load("category_mappings")
    print(f"Type: {type(mappings)}")
    print(f"\n內容：")
    print(mappings)
else:
    print("category_mappings 不存在，請先執行 training pipeline。")

In [ ]:
"""檢視 preprocessor（Pickle）"""
if catalog.exists("preprocessor"):
    preprocessor = catalog.load("preprocessor")
    print(f"Type: {type(preprocessor)}")
    if isinstance(preprocessor, dict):
        print(f"\nKeys: {list(preprocessor.keys())}")
        for k, v in preprocessor.items():
            print(f"\n--- {k} ---")
            print(f"Type: {type(v)}")
            print(v)
    else:
        if hasattr(preprocessor, "get_params"):
            print(f"\nParams:\n{preprocessor.get_params()}")

else:
    print("preprocessor 不存在，請先執行 training pipeline。")

In [ ]:
"""檢視 model（Pickle）"""
if catalog.exists("model"):
    model = catalog.load("model")
    print(f"Type: {type(model)}")
    if hasattr(model, "get_params"):
        print(f"\nParams:\n{model.get_params()}")

    # LightGBM feature importance
    if hasattr(model, "feature_importances_") and hasattr(model, "feature_name_"):
        import pandas as pd
        importance = pd.DataFrame({
            "feature": model.feature_name_,
            "importance": model.feature_importances_,
        }).sort_values("importance", ascending=False)
        print(f"\nTop 20 Feature Importance:")
        display(importance.head(20))
else:
    print("model 不存在，請先執行 training pipeline。")

In [ ]:
"""檢視 best_params（JSON）：Optuna 搜出的最佳超參數"""
if catalog.exists("best_params"):
    import json
    best_params = catalog.load("best_params")
    print(f"Type: {type(best_params)}")
    print(f"\n最佳超參數：")
    print(json.dumps(best_params, indent=2, ensure_ascii=False))
else:
    print("best_params 不存在，請先執行 training pipeline。")

In [ ]:
"""檢視 evaluation_results（JSON）：模型評估結果（mAP、per-product AP 等）"""
if catalog.exists("evaluation_results"):
    import json
    eval_results = catalog.load("evaluation_results")
    print(f"Type: {type(eval_results)}")
    print(f"\n評估結果：")
    print(json.dumps(eval_results, indent=2, ensure_ascii=False))
else:
    print("evaluation_results 不存在，請先執行 training pipeline。")

## Inference Pipeline

In [ ]:
"""檢視 scoring_dataset（Parquet）：推論用的特徵資料集"""
if catalog.exists("scoring_dataset"):
    df = catalog.load("scoring_dataset")
    print(f"Shape: {df.shape}")
    print(f"\nDtypes:\n{df.dtypes}")
    print(f"\nNull 統計:\n{df.isnull().sum()}")
    display(df.head(10))

    # snap_date 分佈
    if "snap_date" in df.columns:
        print(f"\nSnap_date 分佈:\n{df['snap_date'].value_counts().sort_index()}")
    # prod_name 分佈
    if "prod_name" in df.columns:
        print(f"\nProd_name 分佈:\n{df['prod_name'].value_counts()}")
else:
    print("scoring_dataset 不存在，請先執行 inference pipeline。")

In [ ]:
"""檢視 ranked_predictions（Parquet）：排序後的推薦結果"""
if catalog.exists("ranked_predictions"):
    df = catalog.load("ranked_predictions")
    print(f"Shape: {df.shape}")
    print(f"\nDtypes:\n{df.dtypes}")
    print(f"\nNull 統計:\n{df.isnull().sum()}")
    display(df.head(10))

    # 分數分佈統計
    if "score" in df.columns:
        print(f"\n分數分佈：")
        display(df["score"].describe())

    # 各產品平均分數
    if "prod_name" in df.columns and "score" in df.columns:
        avg_score = df.groupby("prod_name")["score"].mean().sort_values(ascending=False)
        print(f"\n各產品平均分數:\n{avg_score}")

    # 排名分佈：確認每位客戶有完整 1~N 排名
    if "rank" in df.columns and "cust_id" in df.columns:
        ranks_per_cust = df.groupby("cust_id")["rank"].agg(["min", "max", "count"])
        print(f"\n每位客戶排名統計：")
        display(ranks_per_cust.describe())
else:
    print("ranked_predictions 不存在，請先執行 inference pipeline。")

## Evaluation Metrics

使用 vectorized decomposition 計算 per-product metrics：在 sorted merged DataFrame 上計算每列的 metric 貢獻值，再以 `groupby("prod_name")` 聚合。

In [ ]:
"""Merged data: predictions + labels"""
if catalog.exists("ranked_predictions") and catalog.exists("label_table"):
    preds = catalog.load("ranked_predictions")
    labels = catalog.load("label_table")

    merged = preds.merge(
        labels[["snap_date", "cust_id", "prod_name", "cust_segment_typ", "label"]],
        on=["snap_date", "cust_id", "prod_name"],
        how="inner",
    )
    print(f"predictions shape: {preds.shape}")
    print(f"labels shape:      {labels.shape}")
    print(f"merged shape:      {merged.shape}")
    print(f"\nmerged columns: {list(merged.columns)}")
    display(merged.head(10))
else:
    print("ranked_predictions 或 label_table 不存在。")

In [ ]:
result = (merged
    .sort_values(["cust_id", "rank"])
    .assign(
        pos=lambda x: x.groupby("cust_id").cumcount() + 1,
        cum_rel=lambda x: x.groupby("cust_id")["label"].cumsum(),
        precision=lambda x: x["cum_rel"] / x["pos"],
        precision_if_rel=lambda x: x["precision"] * x["label"],
    )
)

In [ ]:
import pandas as pd
import numpy as np

ap_by_cust = (
    merged
    .sort_values(["cust_id", "rank"])
    .assign(
        pos=lambda x: x.groupby("cust_id").cumcount() + 1,
        cum_rel=lambda x: x.groupby("cust_id")["label"].cumsum(),
        precision=lambda x: x["cum_rel"] / x["pos"],
        precision_if_rel=lambda x: x["precision"] * x["label"],
    )
    .groupby("prod_name", as_index=False)
    .agg(
        n_rel=("label", "sum"),
        ap_numer=("precision_if_rel", "sum"),
    )
    .assign(
        AP=lambda x: x["ap_numer"] / x["n_rel"]
    )
    .loc[lambda x: x["n_rel"] > 0, ["prod_name", "AP"]]
)

ap_by_cust

In [ ]:
"""Overall metrics: compute_all_metrics 整體結果"""
if "merged" in dir():
    from recsys_tfb.evaluation.metrics import compute_all_metrics

    results = compute_all_metrics(preds, labels)
    print(f"n_queries: {results['n_queries']}")
    print(f"n_excluded_queries: {results['n_excluded_queries']}")
    print(f"\nOverall metrics:")
    for k, v in results["overall"].items():
        print(f"  {k}: {v:.4f}")

In [ ]:
"""Per-product metrics 驗證：確認 vectorized decomposition 修正後結果正確

修正前的 bug：_compute_per_dimension 先 filter 到單一產品再 groupby customer，
導致每個 query group 只有 1 筆 → metrics 全部 = 1.0。

修正後：在完整 merged DataFrame 上計算每列的 metric 貢獻值，
再以 groupby("prod_name") 聚合，得到正確的 per-product metrics。
"""
if "results" in dir():
    print("Per-product metrics（修正後）:")
    per_prod_df = pd.DataFrame(results["per_product"]).T
    display(per_prod_df)

    # 驗證：per-product mAP 不再全部 = 1.0
    all_ones = (per_prod_df["map"] == 1.0).all()
    print(f"\nmAP 全部 = 1.0? {all_ones}  (預期: False)")

    # 與手動 vectorized 計算比較
    print("\n手動驗算 per-product mAP（vectorized precision on label=1 rows）:")
    display(ap_by_cust)

In [ ]:
"""Per-segment metrics 作為對照（segment 是 customer-level 屬性，不會把 group 拆散）"""
if "results" in dir():
    if results["per_segment"]:
        print("Per-segment metrics:")
        per_seg_df = pd.DataFrame(results["per_segment"]).T
        display(per_seg_df)
    else:
        print("labels 中沒有 cust_segment_typ 欄位，無 per_segment 結果。")

In [ ]:
"""Macro / Micro 平均：by_product vs by_segment"""
if "results" in dir():
    print("=== Macro Average ===")
    for dim, vals in results["macro_avg"].items():
        print(f"\n{dim}:")
        for k, v in vals.items():
            print(f"  {k}: {v:.4f}")

    print("\n=== Micro Average ===")
    for dim, vals in results["micro_avg"].items():
        print(f"\n{dim}:")
        for k, v in vals.items():
            print(f"  {k}: {v:.4f}")

    # Per-product 全覽表
    if results["per_product"]:
        print("\n=== Per-product metrics (全部產品) ===")
        per_prod_df = pd.DataFrame(results["per_product"]).T
        display(per_prod_df)